In [0]:

import pandas as pd
import openpyxl

print("Pandas:", pd.__version__)
print("OpenPyXL:", openpyxl.__version__)
print("Dependencias disponibles correctamente")


In [0]:

import pandas as pd

PATH = (
    "/Volumes/workspace/codi_bronze/raw_files/"
    "extracted/2020/"
    "Diccionario BD egresos hospitalario.xlsx"
)

df_cie10 = pd.read_excel(
    PATH,
    sheet_name="codigo CIE-10",
    header=8,
    dtype=str,
    engine="openpyxl"
)

df_cie10.columns = [
    str(c).strip().upper().replace(" ", "_")
    for c in df_cie10.columns
]

df_cie10 = df_cie10.dropna(how="all")

print("Registros:", len(df_cie10))
print("Columnas:", df_cie10.columns.tolist())

display(df_cie10.head(10))


In [0]:

import pandas as pd

COLUMNAS_REQUERIDAS = [
    "CODIGO_SUBCATEGORIA",
    "GLOSA_SUBCATEGORIA",
    "CAPITULO",
    "GLOSA_CAPITULO",
    "CODIGO_GRUPO",
    "GLOSA_GRUPO",
    "CODIGO_CATEGORIA",
    "GLOSA_CATEGORIA"
]

faltantes = [
    c for c in COLUMNAS_REQUERIDAS
    if c not in df_cie10.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas: {faltantes}. "
        f"Disponibles: {df_cie10.columns.tolist()}"
    )

# Normalizar espacios, sin alterar los códigos
for columna in COLUMNAS_REQUERIDAS:
    df_cie10[columna] = (
        df_cie10[columna]
        .astype("string")
        .str.strip()
    )

print("Estructura validada.")
print("Registros:", len(df_cie10))
print("Columnas:", len(df_cie10.columns))

display(df_cie10.head(10))


In [0]:

CODIGO = "CODIGO_SUBCATEGORIA"

# Considerar vacíos y nulos
codigos = df_cie10[CODIGO]

nulos = int(
    (codigos.isna() | codigos.eq("").fillna(False)).sum()
)

# Detectar códigos repetidos
duplicados = df_cie10[
    codigos.notna() & codigos.ne("").fillna(False)
].duplicated(subset=[CODIGO]).sum()

# Valores distintos
unicos = df_cie10[CODIGO].nunique(dropna=True)

print("=" * 55)
print("AUDITORÍA CATÁLOGO CIE-10")
print("=" * 55)

print("Registros:", len(df_cie10))
print("Códigos únicos:", unicos)
print("Códigos nulos/vacíos:", nulos)
print("Filas con códigos duplicados:", duplicados)

if duplicados:
    display(
        df_cie10[
            df_cie10.duplicated(
                subset=[CODIGO],
                keep=False
            )
        ].sort_values(CODIGO).head(30)
    )


In [0]:

from pyspark.sql import functions as F

SILVER_TABLE = (
    "workspace.codi_silver.deis_egresos_historico"
)

if not spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError(
        "La tabla Silver todavía no existe."
    )

# Preparar dimensión temporal
df_dim = spark.createDataFrame(
    df_cie10[COLUMNAS_REQUERIDAS].astype(object)
    .where(pd.notna(df_cie10), None)
    .to_dict("records")
)

df_dim = df_dim.withColumn(
    "codigo_normalizado",
    F.upper(
        F.regexp_replace(
            F.trim(F.col("CODIGO_SUBCATEGORIA")),
            r"[^A-Za-z0-9]",
            ""
        )
    )
)

# Evitar un JOIN ambiguo si existen duplicados
if (
    df_dim.groupBy("codigo_normalizado")
    .count()
    .filter("count > 1")
    .limit(1)
    .count()
):
    raise ValueError(
        "Existen códigos normalizados duplicados. "
        "Revisar antes de continuar."
    )

# Códigos distintos de diagnóstico principal
df_diagnosticos = (
    spark.table(SILVER_TABLE)
    .select("DIAG1")
    .distinct()
    .withColumn(
        "codigo_normalizado",
        F.upper(
            F.regexp_replace(
                F.trim(F.col("DIAG1")),
                r"[^A-Za-z0-9]",
                ""
            )
        )
    )
)

df_cobertura = (
    df_diagnosticos
    .join(
        df_dim.select(
            "codigo_normalizado",
            "GLOSA_SUBCATEGORIA"
        ),
        on="codigo_normalizado",
        how="left"
    )
    .withColumn(
        "tiene_equivalencia",
        F.col("GLOSA_SUBCATEGORIA").isNotNull()
    )
)

display(
    df_cobertura
    .groupBy("tiene_equivalencia")
    .count()
)

# Revisar diagnósticos sin correspondencia
display(
    df_cobertura
    .filter(~F.col("tiene_equivalencia"))
    .limit(30)
)


In [0]:

from pyspark.sql import functions as F

TARGET_TABLE = (
    "workspace.codi_silver.dim_diagnostico_cie10"
)

# Preparar dimensión desde el catálogo validado
df_dimension = (
    df_dim
    .withColumn(
        "_source_system",
        F.lit("DEIS")
    )
    .withColumn(
        "_source_dictionary",
        F.lit("Diccionario BD egresos hospitalario.xlsx")
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

# Comprobar unicidad de la clave normalizada
duplicados = (
    df_dimension
    .groupBy("codigo_normalizado")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicados == 0, (
    "Existen códigos duplicados en la dimensión"
)

# No sobrescribir tablas existentes
if spark.catalog.tableExists(TARGET_TABLE):
    print("La dimensión ya existe. No se modificó.")
else:
    (
        df_dimension.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

    print("Dimensión creada:", TARGET_TABLE)

display(
    spark.table(TARGET_TABLE)
    .select(
        "codigo_normalizado",
        "GLOSA_SUBCATEGORIA",
        "CODIGO_CATEGORIA",
        "GLOSA_CATEGORIA",
        "CODIGO_GRUPO",
        "GLOSA_GRUPO",
        "CAPITULO",
        "GLOSA_CAPITULO"
    )
    .limit(20)
)


In [0]:

TABLE = "workspace.codi_silver.dim_diagnostico_cie10"

df = spark.table(TABLE)

total = df.count()

unicos = (
    df.select("codigo_normalizado")
    .distinct()
    .count()
)

print("=" * 55)
print("VALIDACIÓN DIMENSIÓN CIE-10")
print("=" * 55)
print(f"Registros: {total:,}")
print(f"Códigos únicos: {unicos:,}")
print(f"Duplicados: {total - unicos:,}")

assert total == unicos, (
    "La dimensión contiene claves duplicadas"
)

display(
    df.groupBy(
        "CAPITULO",
        "GLOSA_CAPITULO"
    )
    .agg(
        F.count("*").alias("codigos")
    )
    .orderBy("CAPITULO")
)

print("DIMENSIÓN VALIDADA")
